# 00 — Download APTOS 2019 into Google Drive

**Project:** DR Severity Classification — Member 3 (DenseNet + Backend)

This notebook runs **once**. It downloads the APTOS 2019 dataset from Kaggle and saves
**only the labelled training data** (`train.csv` + `train_images/`) to Google Drive as a single zip:

`MyDrive/DR_project/data/aptos_train.zip`

**Before running:**
1. Kaggle account created and competition rules accepted.
2. Kaggle token added to Colab **Secrets** (🔑 icon in the left sidebar), with *Notebook access* switched ON:
   - Legacy `kaggle.json` → two secrets: `KAGGLE_USERNAME` and `KAGGLE_KEY`
   - New `KGAT_...` token → one secret: `KAGGLE_API_TOKEN`

No GPU is needed for this notebook (Runtime → Change runtime type → CPU is fine).

## 1. Mount Google Drive
A Google permission pop-up will appear — choose your **personal** account and allow access.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DATA_DIR = '/content/drive/MyDrive/DR_project/data'   # permanent storage (Google Drive)
DRIVE_ZIP      = f'{DRIVE_DATA_DIR}/aptos_train.zip'
WORK_DIR       = '/content/aptos'                            # temporary fast disk (wiped when session ends)

os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
print('Drive folder ready:', DRIVE_DATA_DIR)
print('Already downloaded?', os.path.exists(DRIVE_ZIP))

## 2. Check free space in Google Drive
We need about **8 GB** free in Drive. `shutil.disk_usage` reports the Drive quota.

In [ ]:
import shutil
total, used, free = shutil.disk_usage('/content/drive')
print(f'Drive total: {total/1e9:.1f} GB | used: {used/1e9:.1f} GB | free: {free/1e9:.1f} GB')
if free < 9e9:
    print('⚠️  Less than 9 GB free — free up Drive space before continuing.')
else:
    print('✅ Enough space.')

## 3. Load the Kaggle token from Colab Secrets
`userdata.get(name)` reads a secret you stored in the 🔑 panel. The value is put into **environment variables**,
which is where the `kaggle` tool looks for credentials. The token itself is **never printed**.

In [ ]:
from google.colab import userdata

def get_secret(name):
    try:
        return userdata.get(name)
    except Exception:          # secret missing or notebook access not granted
        return None

api_token = get_secret('KAGGLE_API_TOKEN')
username  = get_secret('KAGGLE_USERNAME')
key       = get_secret('KAGGLE_KEY')

if api_token:
    os.environ['KAGGLE_API_TOKEN'] = api_token
    print('✅ Using new-style KAGGLE_API_TOKEN')
elif username and key:
    os.environ['KAGGLE_USERNAME'] = username
    os.environ['KAGGLE_KEY'] = key
    print(f'✅ Using legacy credentials for Kaggle user: {username}')
else:
    raise RuntimeError('No Kaggle secrets found. Add them in the 🔑 Secrets panel and switch ON "Notebook access".')

## 4. Download the competition data (≈ 9.5 GB, to temporary disk)
Skipped automatically if the zip is already in Drive.

- **403 Forbidden** → you have not accepted the competition rules.
- **401 Unauthorized** → the token/secret value is wrong.

In [ ]:
!pip install -q --upgrade kaggle

if os.path.exists(DRIVE_ZIP):
    print('✅ aptos_train.zip already in Drive — skipping download.')
else:
    os.makedirs(WORK_DIR, exist_ok=True)
    !kaggle competitions download -c aptos2019-blindness-detection -p /content
    !ls -lh /content/*.zip

## 5. Extract only the labelled training data
`unzip` with file patterns extracts just `train.csv` and `train_images/*`. We skip `test_images/`
(no labels → can't be used for evaluation).

In [ ]:
if not os.path.exists(DRIVE_ZIP):
    !unzip -q -o /content/aptos2019-blindness-detection.zip "train.csv" "train_images/*" -d {WORK_DIR}
    !ls {WORK_DIR}

## 6. Verify the download
Every row in `train.csv` must have a matching PNG image, and vice versa.
Expected: **3662** rows, **3662** images, **5** classes.

In [ ]:
import pandas as pd

if not os.path.exists(DRIVE_ZIP):
    df = pd.read_csv(f'{WORK_DIR}/train.csv')
    image_files = {f[:-4] for f in os.listdir(f'{WORK_DIR}/train_images') if f.endswith('.png')}
    csv_ids = set(df['id_code'])

    print('CSV columns   :', list(df.columns))
    print('CSV rows      :', len(df))
    print('PNG images    :', len(image_files))
    print('Missing images:', len(csv_ids - image_files))
    print('Extra images  :', len(image_files - csv_ids))
    print('\nImages per class:')
    print(df['diagnosis'].value_counts().sort_index().to_string())

    assert len(csv_ids - image_files) == 0 and len(image_files - csv_ids) == 0, 'CSV and images do not match!'
    print('\n✅ Dataset verified')

## 7. Save one zip to Google Drive
`zip -0` = **store without compression** (PNGs are already compressed, so this is fast and loses nothing).
Copying **one** big file to Drive is much faster and more reliable than 3,662 small files.

In [ ]:
if not os.path.exists(DRIVE_ZIP):
    !cd {WORK_DIR} && zip -0 -q -r /content/aptos_train.zip train.csv train_images
    !cp /content/aptos_train.zip "{DRIVE_ZIP}"
    print('✅ Copied to Drive')

!ls -lh "{DRIVE_ZIP}"

## 8. Done — how later notebooks will use it
Every future training notebook starts with:
```python
!cp /content/drive/MyDrive/DR_project/data/aptos_train.zip /content/
!unzip -q /content/aptos_train.zip -d /content/aptos
```
Then images are read from the fast local disk at `/content/aptos/train_images/`.